# 使用 Optuna 进行超参数调优

Github 仓库: https://github.com/araffin/tools-for-robotic-rl-icra2022

Optuna: https://github.com/optuna/optuna

Stable-Baselines3: https://github.com/DLR-RM/stable-baselines3

文档: https://stable-baselines3.readthedocs.io/en/master/

SB3 Contrib: https://github.com/Stable-Baselines-Team/stable-baselines3-contrib

RL Baselines3 zoo: https://github.com/DLR-RM/rl-baselines3-zoo

[RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo) 是一个基于 Stable-Baselines3 的预训练强化学习智能体合集。

它还提供了训练和评估智能体、调优超参数以及录制视频的基础脚本。


## 简介

在本笔记本中，你将了解超参数调优的重要性。你会先手动尝试优化参数，然后我们再看如何用 Optuna 自动完成搜索。


## 使用 Pip 安装依赖和 Stable Baselines3

完整依赖列表见 [README](https://github.com/DLR-RM/stable-baselines3)。


```
pip install stable-baselines3[extra]
```

In [1]:
!pip install stable-baselines3

zsh:1: command not found: pip


In [2]:
# 可选：安装 SB3 contrib 以使用更多算法
!pip install sb3-contrib

zsh:1: command not found: pip


In [3]:
# 最后一部分做超参数调优时会用到 Optuna
!pip install optuna

zsh:1: command not found: pip


## 导入

In [4]:
import gym
import numpy as np

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


首先需要导入强化学习模型，查阅文档了解哪些算法适用于哪类问题

In [5]:
from stable_baselines3 import PPO, A2C, SAC, TD3, DQN

In [6]:
# 来自 contrib 仓库的算法
# https://github.com/Stable-Baselines-Team/stable-baselines3-contrib
from sb3_contrib import QRDQN, TQC

In [7]:
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy

# 第一部分：调优超参数的重要性



与监督学习相比，深度强化学习对超参数的选择要敏感得多，例如学习率、神经元数量、层数、优化器等。

超参数选得不好会导致收敛效果差或不稳定。不同随机种子（用于初始化网络权重和环境）之间的性能差异让这个问题更加棘手。

除了超参数，选择合适的算法同样重要。我们将在简单的 Pendulum 任务上演示这一点。

参见 [gym 文档](https://gym.openai.com/envs/Pendulum-v0/)：“倒立摆起摆问题是控制领域的经典问题。在这个版本中，摆从随机位置开始，目标是把它摆起来并保持直立。”


先用 PPO 和 4000 步（20 个回合）的小预算试试：

In [8]:
env_id = "Pendulum-v1"
# 仅用于评估的环境
eval_envs = make_vec_env(env_id, n_envs=10)
# 训练 4000 个时间步
budget_pendulum = 4000

### PPO

In [9]:
ppo_model = PPO("MlpPolicy", env_id, seed=0, verbose=0).learn(budget_pendulum)

In [10]:
mean_reward, std_reward = evaluate_policy(ppo_model, eval_envs, n_eval_episodes=100, deterministic=True)

print(f"PPO Mean episode reward: {mean_reward:.2f} +/- {std_reward:.2f}")

PPO Mean episode reward: -1137.87 +/- 215.74


### A2C

In [11]:
# 定义并训练一个 A2C 模型
a2c_model = A2C("MlpPolicy", env_id, seed=0, verbose=0).learn(budget_pendulum)

In [12]:
# 评估训练好的 A2C 模型
mean_reward, std_reward = evaluate_policy(a2c_model, eval_envs, n_eval_episodes=100, deterministic=True)

print(f"A2C Mean episode reward: {mean_reward:.2f} +/- {std_reward:.2f}")

A2C Mean episode reward: -1238.71 +/- 237.26


两者都远未解决该环境（解决时平均奖励约为 -200）。
现在试试离策略（off-policy）算法：

### 让 PPO 训练更久？

训练更久也许会有帮助？

你可以试试 10 倍预算，但对 A2C/PPO 来说，训练更久帮助不大，真正需要的是找到更好的超参数。

In [13]:
# 训练更久
new_budget = 10 * budget_pendulum

ppo_model = PPO("MlpPolicy", env_id, seed=0, verbose=0).learn(new_budget)

In [14]:
mean_reward, std_reward = evaluate_policy(ppo_model, eval_envs, n_eval_episodes=100, deterministic=True)

print(f"PPO Mean episode reward: {mean_reward:.2f} +/- {std_reward:.2f}")

PPO Mean episode reward: -1181.87 +/- 311.53


### PPO - 调优后的超参数

借助 Optuna，我们确实可以调出超参数并找到可行的解（来自 [RL Zoo](https://github.com/DLR-RM/rl-baselines3-zoo/blob/master/hyperparams/ppo.yml)）：

In [15]:
tuned_params = {
    "gamma": 0.9,
    "use_sde": True,
    "sde_sample_freq": 4,
    "learning_rate": 1e-3,
}

# budget = 10 * budget_pendulum
ppo_tuned_model = PPO("MlpPolicy", env_id, seed=1, verbose=1, **tuned_params).learn(50_000, log_interval=5)

Using cpu device
Creating environment from the given name 'Pendulum-v1'
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 200        |
|    ep_rew_mean          | -1.21e+03  |
| time/                   |            |
|    fps                  | 7356       |
|    iterations           | 5          |
|    time_elapsed         | 1          |
|    total_timesteps      | 10240      |
| train/                  |            |
|    approx_kl            | 0.02233348 |
|    clip_fraction        | 0.178      |
|    clip_range           | 0.2        |
|    entropy_loss         | -2.6       |
|    explained_variance   | 0.824      |
|    learning_rate        | 0.001      |
|    loss                 | 11.4       |
|    n_updates            | 40         |
|    policy_gradient_loss | -0.0148    |
|    std                  | 0.926      |
|    value_loss           | 34.

In [16]:
mean_reward, std_reward = evaluate_policy(ppo_tuned_model, eval_envs, n_eval_episodes=100, deterministic=True)

print(f"Tuned PPO Mean episode reward: {mean_reward:.2f} +/- {std_reward:.2f}")

Tuned PPO Mean episode reward: -179.56 +/- 96.34


注意：如果在简单的 MountainCarContinuous 环境上使用未调优超参数的 SAC，你会遇到一些问题，见 https://github.com/rail-berkeley/softlearning/issues/76

即使对 SOTA 算法来说，简单环境也可能很有挑战。

# 第二部分：研究生下降法（Grad Student Descent）


### 挑战（10 分钟）：“研究生下降法”
挑战内容：在 20 000 训练步的有限预算内，为 `CartPole-v1` 上的 A2C 找到最佳超参数（性能最高）。


`CartPole-v1` 的最高奖励：500

这组超参数应当在不同随机种子下都有效。

In [17]:
budget = 20_000

#### 基线：默认超参数

In [18]:
eval_envs_cartpole = make_vec_env("CartPole-v1", n_envs=10)

In [19]:
model = A2C("MlpPolicy", "CartPole-v1", seed=0, verbose=1).learn(budget)

Using cpu device
Creating environment from the given name 'CartPole-v1'
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 47       |
|    ep_rew_mean        | 47       |
| time/                 |          |
|    fps                | 5397     |
|    iterations         | 100      |
|    time_elapsed       | 0        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -0.596   |
|    explained_variance | -0.0575  |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | 1.09     |
|    value_loss         | 9.74     |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 54.3     |
|    ep_rew_mean        | 54.3     |
| time/                 |          |
|    fps                | 5586     |


In [20]:
mean_reward, std_reward = evaluate_policy(model, eval_envs_cartpole, n_eval_episodes=50, deterministic=True)

print(f"mean_reward:{mean_reward:.2f} +/- {std_reward:.2f}")

mean_reward:175.58 +/- 64.71


**你的目标是超过这个基线，并尽量接近最优分数 500**

开始调参！

In [21]:
import torch.nn as nn

In [22]:
policy_kwargs = dict(
    net_arch=[
      dict(vf=[64, 64], pi=[64, 64]), # actor/critic 的网络结构
    ],
    activation_fn=nn.Tanh,
)

hyperparams = dict(
    n_steps=5, # 每次更新策略前收集数据的步数
    learning_rate=7e-4,
    gamma=0.99, # 折扣因子
    max_grad_norm=0.5, # 梯度裁剪的最大值
    ent_coef=0.0, # 损失计算中的熵系数
)

model = A2C("MlpPolicy", "CartPole-v1", seed=0, verbose=1, **hyperparams).learn(budget)

Using cpu device
Creating environment from the given name 'CartPole-v1'
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 47       |
|    ep_rew_mean        | 47       |
| time/                 |          |
|    fps                | 5458     |
|    iterations         | 100      |
|    time_elapsed       | 0        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -0.596   |
|    explained_variance | -0.0575  |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | 1.09     |
|    value_loss         | 9.74     |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 54.3     |
|    ep_rew_mean        | 54.3     |
| time/                 |          |
|    fps                | 5635     |


In [23]:
mean_reward, std_reward = evaluate_policy(model, eval_envs_cartpole, n_eval_episodes=50, deterministic=True)

print(f"mean_reward:{mean_reward:.2f} +/- {std_reward:.2f}")

mean_reward:175.98 +/- 66.86


提示 - 推荐的超参数范围

```python
gamma = trial.suggest_float("gamma", 0.9, 0.99999, log=True)
max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 5.0, log=True)
# 从 2**3 = 8 到 2**10 = 1024
n_steps = 2 ** trial.suggest_int("exponent_n_steps", 3, 10)
learning_rate = trial.suggest_float("lr", 1e-5, 1, log=True)
ent_coef = trial.suggest_float("ent_coef", 0.00000001, 0.1, log=True)
# net_arch tiny: {"pi": [64], "vf": [64]}
# net_arch default: {"pi": [64, 64], "vf": [64, 64]}
# activation_fn = nn.Tanh / nn.ReLU
```

# 第三部分：自动超参数调优





这一部分我们将编写一个能自动搜索最佳超参数的脚本。

### 导入

In [24]:
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from optuna.visualization import plot_optimization_history, plot_param_importances

### 配置

In [25]:
N_TRIALS = 100  # 最大试验次数
N_JOBS = 1 # 并行运行的任务数
N_STARTUP_TRIALS = 5  # 前 N_STARTUP_TRIALS 次试验后停止随机采样
N_EVALUATIONS = 2  # 训练过程中的评估次数
N_TIMESTEPS = int(2e4)  # 训练预算
EVAL_FREQ = int(N_TIMESTEPS / N_EVALUATIONS)
N_EVAL_ENVS = 5
N_EVAL_EPISODES = 10
TIMEOUT = int(60 * 15)  # 15 分钟

ENV_ID = "CartPole-v1"

DEFAULT_HYPERPARAMS = {
    "policy": "MlpPolicy",
    "env": ENV_ID,
}

### 练习（5 分钟）：定义搜索空间

In [26]:
from typing import Any, Dict
import torch
import torch.nn as nn

def sample_a2c_params(trial: optuna.Trial) -> Dict[str, Any]:
    """
    A2C 超参数采样器。

    :param trial: Optuna trial 对象
    :return: 为该 trial 采样得到的超参数。
    """
    # 折扣因子，取值在 0.9 到 0.9999 之间
    gamma = 1.0 - trial.suggest_float("gamma", 0.0001, 0.1, log=True)
    max_grad_norm = trial.suggest_float("max_grad_norm", 0.3, 5.0, log=True)
    # 8, 16, 32, ... 1024
    n_steps = 2 ** trial.suggest_int("exponent_n_steps", 3, 10)

    ### 在此编写你的代码
    # TODO:
    # - 定义学习率搜索空间 [1e-5, 1]（对数尺度） -> `suggest_float`
    # - 定义网络结构搜索空间 ["tiny", "small"] -> `suggest_categorical`
    # - 定义激活函数搜索空间 ["tanh", "relu"]
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1, log=True)
    net_arch = trial.suggest_categorical("net_arch", ["tiny", "small"])
    activation_fn = trial.suggest_categorical("activation_fn", ["tanh", "relu"])

    ### 你的代码到此结束

    # 记录换算后的真实值
    trial.set_user_attr("gamma_", gamma)
    trial.set_user_attr("n_steps", n_steps)

    net_arch = [
        {"pi": [64], "vf": [64]}  # tiny
        if net_arch == "tiny"
        else {"pi": [64, 64], "vf": [64, 64]}  # small
    ]

    activation_fn = {"tanh": nn.Tanh, "relu": nn.ReLU}[activation_fn]

    return {
        "n_steps": n_steps,
        "gamma": gamma,
        "learning_rate": learning_rate,
        "max_grad_norm": max_grad_norm,
        "policy_kwargs": {
            "net_arch": net_arch,
            "activation_fn": activation_fn,
        },
    }

### 定义目标函数

首先定义一个自定义回调，把周期性评估的结果报告给 Optuna：

In [27]:
from stable_baselines3.common.callbacks import EvalCallback

class TrialEvalCallback(EvalCallback):
    """
    用于评估并报告 trial 的回调。

    :param eval_env: 评估环境
    :param trial: Optuna trial 对象
    :param n_eval_episodes: 评估回合数
    :param eval_freq: 回调每被调用 ``eval_freq`` 次评估一次智能体。
    :param deterministic: 评估时使用随机策略还是确定性策略。
    :param verbose:
    """

    def __init__(
        self,
        eval_env: gym.Env,
        trial: optuna.Trial,
        n_eval_episodes: int = 5,
        eval_freq: int = 10000,
        deterministic: bool = True,
        verbose: int = 0,
    ):

        super().__init__(
            eval_env=eval_env,
            n_eval_episodes=n_eval_episodes,
            eval_freq=eval_freq,
            deterministic=deterministic,
            verbose=verbose,
        )
        self.trial = trial
        self.eval_idx = 0
        self.is_pruned = False

    def _on_step(self) -> bool:
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            # 评估策略（在父类中完成）
            super()._on_step()
            self.eval_idx += 1
            # 向 Optuna 报告结果
            self.trial.report(self.last_mean_reward, self.eval_idx)
            # 必要时剪枝该 trial
            if self.trial.should_prune():
                self.is_pruned = True
                return False
        return True

### 练习（10 分钟）：定义目标函数

接着定义目标函数，它负责采样超参数、创建模型，然后把结果返回给 Optuna

In [28]:
def objective(trial: optuna.Trial) -> float:
    """
    Optuna 用来评估一组配置（即一组超参数）的目标函数。

    给定一个 trial 对象，它会采样超参数、进行评估，
    并报告结果（训练后的平均回合奖励）

    :param trial: Optuna trial 对象
    :return: 训练后的平均回合奖励
    """

    kwargs = DEFAULT_HYPERPARAMS.copy()
    ### 在此编写你的代码
    # TODO:
    # 1. 采样超参数并更新默认关键字参数： `kwargs.update(other_params)`
    # 2. 创建评估环境
    # 3. 创建 `TrialEvalCallback`

    # 1. 采样超参数并更新关键字参数
    other_params = sample_a2c_params(trial)
    kwargs.update(other_params)

    # 创建强化学习模型
    model = A2C(**kwargs)

    # 2. 用 `make_vec_env`、`ENV_ID` 和 `N_EVAL_ENVS` 创建评估环境
    eval_envs = make_vec_env(ENV_ID, n_envs=N_EVAL_ENVS)
    # 3. 创建上面定义的 `TrialEvalCallback` 回调，它每隔 `EVAL_FREQ` 步
    # 用 `N_EVAL_EPISODES` 个回合评估一次并报告性能
    # TrialEvalCallback 的签名：
    # TrialEvalCallback(eval_env, trial, n_eval_episodes, eval_freq, deterministic, verbose)
    eval_callback = TrialEvalCallback(
        eval_envs,
        trial,
        n_eval_episodes=N_EVAL_EPISODES,
        eval_freq=EVAL_FREQ,
        deterministic=True,
    )

    ### 你的代码到此结束

    nan_encountered = False
    try:
        # 训练模型
        model.learn(N_TIMESTEPS, callback=eval_callback)
    except AssertionError as e:
        # 随机采样的超参数有时会产生 NaN
        print(e)
        nan_encountered = True
    finally:
        # 释放内存
        model.env.close()
        eval_envs.close()

    # 告诉优化器该 trial 失败了
    if nan_encountered:
        return float("nan")

    if eval_callback.is_pruned:
        raise optuna.exceptions.TrialPruned()

    return eval_callback.last_mean_reward

### 优化循环

In [29]:
import torch as th

# 将 pytorch 线程数设为 1 以加快训练
th.set_num_threads(1)
# 选择采样器，可以是 random、TPESampler、CMAES 等
sampler = TPESampler(n_startup_trials=N_STARTUP_TRIALS)
# 用掉最大预算的 1/3 之前不剪枝
pruner = MedianPruner(
    n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=N_EVALUATIONS // 3
)
# 创建 study 并开始超参数优化
study = optuna.create_study(sampler=sampler, pruner=pruner, direction="maximize")

try:
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=N_JOBS, timeout=TIMEOUT)
except KeyboardInterrupt:
    pass

print("Number of finished trials: ", len(study.trials))

print("Best trial:")
trial = study.best_trial

print(f"  Value: {trial.value}")

print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

print("  User attrs:")
for key, value in trial.user_attrs.items():
    print(f"    {key}: {value}")

# 写出报告
study.trials_dataframe().to_csv("study_results_a2c_cartpole.csv")

fig1 = plot_optimization_history(study)
fig2 = plot_param_importances(study)

fig1.show()
fig2.show()

[I 2026-10-08 10:46:20,375] A new study created in memory with name: no-name-432a1187-b6bf-48e4-8a65-ad0ecc7ae97b
/Users/zhangyifan/LearningProjects/Huggin-Face-RL-Learning/.venv/lib/python3.12/site-packages/stable_baselines3/common/policies.py:486: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(
[I 2026-10-08 10:46:22,053] Trial 0 finished with value: 9.5 and parameters: {'gamma': 0.010407728147467817, 'max_grad_norm': 0.4191132438356876, 'exponent_n_steps': 6, 'learning_rate': 0.20325048954701572, 'net_arch': 'small', 'activation_fn': 'tanh'}. Best is trial 0 with value: 9.5.
[I 2026-10-08 10:46:23,564] Trial 1 finished with value: 500.0 and parameters: {'gamma': 0.09448161619460885, 'max_grad_norm': 0.40152548080233846, 'exponent_n_steps': 5, 'learning_rate': 0.016976072373238782, 'net_arch': 'tiny', 'ac

Number of finished trials:  100
Best trial:
  Value: 500.0
  Params: 
    gamma: 0.09448161619460885
    max_grad_norm: 0.40152548080233846
    exponent_n_steps: 5
    learning_rate: 0.016976072373238782
    net_arch: tiny
    activation_fn: relu
  User attrs:
    gamma_: 0.9055183838053912
    n_steps: 32


完整示例: https://github.com/DLR-RM/rl-baselines3-zoo

# 总结

本笔记本中我们学到了：
- 好的超参数有多重要
- 如何用 optuna 自动搜索超参数
